In [ ]:
from hydra import initialize, compose
from hydra.core.global_hydra import GlobalHydra
from IPython.display import display, HTML
from pathlib import Path

from lensieve.image import load_image
from lensieve.models.model_manager import get_model_manager
from lensieve.data.data_store import DataStore
from lensieve.agent.photo_agent import PhotoAgent
from lensieve.logging_config import setup_logging

import warnings

warnings.filterwarnings(
    "ignore",
    message=".*local_dir_use_symlinks.*",
    category=UserWarning,
)

In [ ]:
GlobalHydra.instance().clear()

with initialize(config_path="../configs", version_base=None):
    cfg = compose(config_name="agent_config")

setup_logging(root=cfg.root, app_name="agent_loop", verbose=False)
model_manager = get_model_manager(cfg)
data_store = DataStore("../data/large_samsung")

agent = PhotoAgent(model_manager=model_manager, 
                   data_store=data_store, 
                   northern_hemisphere=cfg.agent.northern_hemisphere,
                   max_image_results=cfg.agent.tools.search_photos.max_results,
                   max_metadata_results=cfg.agent.tools.query_metadata.max_results,
                  )

In [ ]:
# parameters
MAX_WIDTH = 200   # px
IMAGES_PER_ROW = 5

def render_images(res):
    rows = []
    current_row = []

    for i, row in enumerate(res):
        path = Path(row.path)
        score = row.score

        try:
            img = load_image(data_store.root / path)

            # resize (keep aspect ratio)
            w, h = img.size
            scale = MAX_WIDTH / w
            img = img.resize((int(w * scale), int(h * scale)))

            # convert to base64 for inline display
            import io, base64
            buffer = io.BytesIO()
            img.save(buffer, format="JPEG")
            img_b64 = base64.b64encode(buffer.getvalue()).decode()

            html = f"""
            <div style="text-align:center; margin:5px;">
                <img src="data:image/jpeg;base64,{img_b64}" />
                <div>{path}</div>
                <div>{score:.4f}</div>
            </div>
            """
            current_row.append(html)

        except Exception as e:
            current_row.append(f"<div>Error: {e}</div>")

        if len(current_row) == IMAGES_PER_ROW:
            rows.append(current_row)
            current_row = []

    if current_row:
        rows.append(current_row)

    # build table
    table_html = "<table>"
    for row in rows:
        table_html += "<tr>" + "".join(f"<td>{cell}</td>" for cell in row) + "</tr>"
    table_html += "</table>"

    display(HTML(table_html))

In [ ]:

#results = agent.run_once("young happy woman")
results = agent.run_once("How many unique images are there in the last 4 years grouped by image format and aspect ratio?") # 7B returns only the top counts
#results = agent.run_once("How many images are there by orientation taken during the last 4 years?")  # May return future, does not output dates - 7B fails to generate a tool call
#results = agent.run_once("In what month were the most images taken?") # 7B fails to generate a tool call
#results = agent.run_once("In what month of what year were the most images taken?") # 7B works, but needs a repair
#results = agent.run_once("Give the top month/year combinations when the most images were taken")
#results = agent.run_once("What is the average number of photos taken per day on days when photos were taken?") # 7B works, but needs a repair
#results = agent.run_once("Are there panorama photos?")
#results = agent.run_once("How many panorama photos are there?")  # 7B model fails to generate a tool call
#results = agent.run_once("Are there any heic images?")  # Correctly maps heic -> heif!
#results = agent.run_once("Based on camera makes and models, when was the phone switched?")  # Wow!
#results = agent.run_once("How many circular images were taken last year?")  # Uses 1:1 - obviously an overreach, but whatever...
#results = agent.run_once("How many images of boxes are there?")   # Knows how to answer!
#results = agent.run_once("How many beautiful images are there?")  # Knows how to answer!
if type(results) == str:
    print(results)
else:
    render_images(results.hits)